# Breast Cancer Detection Using Deep Learning with ROI-Based Training

## CBIS-DDSM Mammography Classification Pipeline (ROI Images)

**Reference:** Modified DenseNet-121 with Transfer Learning (ILB-BCD Algorithm)

**Dataset:** CBIS-DDSM Region of Interest (ROI) Cropped Images

**Key Difference from V11:** This pipeline uses cropped ROI images centered on lesions rather than full mammograms. ROI-based training focuses the model on diagnostically relevant regions, potentially reducing noise from irrelevant breast tissue.

---

### Pipeline Structure

1. Environment Configuration
2. Data Loading (ROI-specific cache)
3. Dataset Pipeline Construction
4. Model Architecture
5. Three-Stage Progressive Training
6. Evaluation and Analysis

## 1. Environment Setup

Configure runtime environment, mount storage, and set up GPU memory management.

In [ ]:
# Environment Setup and GPU Configuration

import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
try:
 from google.colab import drive
 drive.mount('/content/drive')
 RUNTIME_ENV = "colab"
 print("Google Drive mounted successfully")
except ImportError:
 RUNTIME_ENV = "local"
 print("Running in local environment")

# Install pydicom if needed
if RUNTIME_ENV == "colab":
 import subprocess
 subprocess.run(['pip', 'install', 'pydicom', '-q'], capture_output=True)

# Core imports
import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

# Image processing
import cv2
import pydicom
from PIL import Image
from tqdm import tqdm

# Deep learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

# Evaluation metrics
from sklearn.metrics import (
 roc_auc_score, roc_curve, classification_report, confusion_matrix,
 accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# GPU Configuration
physical_gpus = tf.config.list_physical_devices('GPU')
GPU_AVAILABLE = len(physical_gpus) > 0
GPU_MEMORY_GB = 16

if GPU_AVAILABLE:
 for gpu in physical_gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 GPU_NAME = tf.config.experimental.get_device_details(physical_gpus[0]).get('device_name', 'Unknown')
else:
 GPU_NAME = "None"

print("-" * 60)
print("ENVIRONMENT CONFIGURATION")
print("-" * 60)
print(f"Runtime: {RUNTIME_ENV.upper()}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"GPU Available: {GPU_AVAILABLE}")
if GPU_AVAILABLE:
 print(f"GPU Name: {GPU_NAME}")
print("-" * 60)

## 2. Configuration Parameters

Define hyperparameters and paths. This configuration uses ROI-specific cache files containing cropped lesion images.

**ROI Advantages:**
- Reduced image noise from non-lesion breast tissue
- Smaller effective dataset size per image
- Model focuses on diagnostically relevant features
- Faster training due to more focused learning signal

In [ ]:
class ExperimentConfig:
 """
 Configuration for ROI-based breast cancer detection experiment.
 
 Uses cropped Region of Interest images centered on lesions
 rather than full mammogram images.
 """
 
 def __init__(self, gpu_memory_gb=16):
 self.gpu_memory_gb = gpu_memory_gb
 self.runtime_env = RUNTIME_ENV
 
 # Data Paths - ROI-specific
 if self.runtime_env == "colab":
 self.base_path = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.dicom_path = self.base_path / 'CBIS-DDSM'
 self.csv_path = self.base_path / 'csv'
 # ROI-specific cache directory
 self.cache_path = self.base_path / 'preprocessed_cache_roi'
 self.checkpoint_path = self.base_path / 'checkpoints_roi'
 self.results_path = self.base_path / 'results_roi'
 else:
 self.base_path = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.dicom_path = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.csv_path = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.cache_path = self.base_path / 'preprocessed_cache_roi'
 self.checkpoint_path = self.base_path / 'checkpoints_roi'
 self.results_path = self.base_path / 'results_roi'
 
 # CSV file paths
 self.calc_train_csv = self.csv_path / 'calc_case_description_train_set.csv'
 self.calc_test_csv = self.csv_path / 'calc_case_description_test_set.csv'
 self.mass_train_csv = self.csv_path / 'mass_case_description_train_set.csv'
 self.mass_test_csv = self.csv_path / 'mass_case_description_test_set.csv'
 
 # Image Configuration
 self.image_size = (224, 224)
 self.image_channels = 3
 
 # CLAHE Preprocessing
 self.use_clahe = True
 self.clahe_clip_limit = 2.0
 self.clahe_tile_size = (8, 8)
 
 # Batch Sizes and Model Configuration
 if gpu_memory_gb >= 16:
 self.batch_size = 32
 self.batch_size_finetune = 16
 self.batch_size_inference = 64
 self.dense_units = 2048
 self.ensemble_size = 3
 self.use_mixed_precision = True
 else:
 self.batch_size = 8
 self.batch_size_finetune = 4
 self.batch_size_inference = 16
 self.dense_units = 1536
 self.ensemble_size = 1
 self.use_mixed_precision = False
 
 self.ensemble_seeds = [42, 123, 456][:self.ensemble_size]
 
 # Training Parameters
 self.stage1_epochs = 30
 self.stage1_lr = 1e-3
 
 self.stage2_epochs = 50
 self.stage2_lr = 3e-4
 self.stage2_frozen_layers = 150 # Freeze first 150 layers in stage 2
 
 self.stage3_epochs = 100
 self.stage3_lr = 1e-4
 
 # Learning rate warmup
 self.use_lr_warmup = True
 self.warmup_epochs = 5
 self.warmup_start_lr = 1e-7
 
 # Data split
 self.train_split = 0.70
 self.val_split = 0.15
 self.test_split = 0.15
 
 # Regularization
 self.l1_regularization = 0.0
 self.l2_regularization = 5e-4
 self.dropout_rate = 0.35
 self.label_smoothing = 0.1
 self.gradient_clip_norm = 1.0
 
 # Class Weighting
 self.use_class_weight = True
 self.malignant_weight_multiplier = 2.5
 self.class_weights = None # Will be computed after loading data
 
 # Focal Loss
 self.use_focal_loss = True
 self.focal_gamma = 2.0
 self.focal_alpha = 0.7
 
 # Architecture
 if gpu_memory_gb >= 16:
 self.freeze_layers_stage3 = 80
 else:
 self.freeze_layers_stage3 = 150
 
 # Callbacks
 self.early_stop_patience = 30
 self.lr_reduce_patience = 12
 self.lr_reduce_factor = 0.5
 self.min_epochs_stage1 = 15
 self.min_epochs_stage2 = 25
 self.min_epochs_stage3 = 50
 
 self.min_train_auc_threshold = 0.55
 self.dead_model_check_epoch = 10
 
 # Test-Time Augmentation
 self.use_tta = True
 self.tta_augmentations = 4
 
 # Performance
 self.prefetch_buffer = tf.data.AUTOTUNE
 self.shuffle_buffer = 1000
 self.num_parallel_calls = tf.data.AUTOTUNE
 
 self._create_directories()
 
 def _create_directories(self):
 """Create output directories."""
 for path in [self.cache_path, self.checkpoint_path, self.results_path]:
 path.mkdir(parents=True, exist_ok=True)
 
 def display(self):
 """Display configuration summary."""
 total_epochs = (self.stage1_epochs + self.stage2_epochs + self.stage3_epochs)
 total_training_epochs = total_epochs * self.ensemble_size
 
 print("-" * 60)
 print("EXPERIMENT CONFIGURATION (ROI-BASED)")
 print("-" * 60)
 print(f"GPU Memory: {self.gpu_memory_gb} GB")
 print(f"Mixed Precision: {self.use_mixed_precision}")
 print("-" * 60)
 print("Model Architecture:")
 print(f" Dense Units: {self.dense_units}")
 print(f" Dropout Rate: {self.dropout_rate}")
 print(f" L2 Regularization: {self.l2_regularization}")
 print("-" * 60)
 print("Training Configuration:")
 print(f" Batch Size: {self.batch_size}")
 print(f" Finetune Batch: {self.batch_size_finetune}")
 print(f" Ensemble Models: {self.ensemble_size}")
 print(f" Stage 1: {self.stage1_epochs} epochs @ LR={self.stage1_lr}")
 print(f" Stage 2: {self.stage2_epochs} epochs @ LR={self.stage2_lr}")
 print(f" Stage 3: {self.stage3_epochs} epochs @ LR={self.stage3_lr}")
 print(f" Total Epochs: {total_training_epochs}")
 print("-" * 60)
 print("Paths (ROI-specific):")
 print(f" Cache: {self.cache_path}")
 print(f" Checkpoints: {self.checkpoint_path}")
 print(f" Results: {self.results_path}")
 print("-" * 60)


# Initialize configuration
config = ExperimentConfig(gpu_memory_gb=GPU_MEMORY_GB)
config.display()

# Enable mixed precision
if GPU_AVAILABLE and config.use_mixed_precision:
 try:
 policy = tf.keras.mixed_precision.Policy('mixed_float16')
 tf.keras.mixed_precision.set_global_policy(policy)
 print("Mixed precision (FP16) enabled")
 except Exception as e:
 print(f"Mixed precision not available: {e}")
 config.use_mixed_precision = False

## 3. ROI Data Preprocessing

This section handles preprocessing of ROI (Region of Interest) cropped images from CBIS-DDSM. If cached data exists, loading proceeds directly. Otherwise, ROI images are extracted from the dataset structure.

**ROI Image Characteristics:**
- Cropped around lesion area with some surrounding context
- Variable original sizes, resized to 224x224
- Contains both mass and calcification lesions

In [ ]:
def apply_clahe(image, clip_limit=2.0, tile_size=(8, 8)):
 """
 Apply Contrast Limited Adaptive Histogram Equalization.
 
 Args:
 image: Input grayscale image (uint8 or uint16)
 clip_limit: Threshold for contrast limiting
 tile_size: Size of grid for histogram equalization
 
 Returns:
 CLAHE enhanced image normalized to [0, 1]
 """
 if image.dtype!= np.uint8:
 image = ((image - image.min()) / (image.max() - image.min() + 1e-8) * 255).astype(np.uint8)
 
 clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
 enhanced = clahe.apply(image)
 
 return enhanced.astype(np.float32) / 255.0


def find_dicom_file(case_dir_name, base_path):
 """
 Find DICOM file in case directory.
 
 CBIS-DDSM structure: case_dir / date_folder / series_folder / file.dcm
 The CSV contains full paths, but only the the case directory name
 (first part of the path) to locate the DICOM file.
 
 Args:
 case_dir_name: Name of the case directory (first part of CSV path)
 base_path: Base path to DICOM directory
 
 Returns:
 Path to DICOM file or None
 """
 case_path = base_path / case_dir_name
 
 if not case_path.exists():
 return None
 
 try:
 # Navigate through the 3-level nesting: case / date / series / file.dcm
 for date_folder in case_path.iterdir():
 if date_folder.is_dir():
 for series_folder in date_folder.iterdir():
 if series_folder.is_dir():
 for dcm_file in series_folder.iterdir():
 if dcm_file.suffix == '.dcm':
 return dcm_file
 except Exception:
 pass
 
 return None


def load_dicom_image(dicom_path, target_size=(224, 224), use_clahe=True):
 """
 Load and preprocess DICOM image.
 
 Args:
 dicom_path: Path to DICOM file
 target_size: Output image dimensions
 use_clahe: Whether to apply CLAHE enhancement
 
 Returns:
 Preprocessed image array of shape (H, W, 3) or None on failure
 """
 try:
 dcm = pydicom.dcmread(str(dicom_path))
 image = dcm.pixel_array.astype(np.float32)
 
 # Handle photometric interpretation
 if hasattr(dcm, 'PhotometricInterpretation'):
 if dcm.PhotometricInterpretation == 'MONOCHROME1':
 image = image.max() - image
 
 # Normalize to 0-255 range for CLAHE
 image = ((image - image.min()) / (image.max() - image.min() + 1e-8) * 255).astype(np.uint8)
 
 # Apply CLAHE
 if use_clahe:
 image = apply_clahe(image)
 else:
 image = image.astype(np.float32) / 255.0
 
 # Resize
 image = cv2.resize(image, target_size, interpolation=cv2.INTER_LANCZOS4)
 
 # Convert to 3-channel
 image = np.stack([image] * 3, axis=-1)
 
 return image
 
 except Exception as e:
 return None


def preprocess_roi_dataset(config, csv_files, split_name='train'):
 """
 Preprocess ROI images from CBIS-DDSM dataset.
 
 The CSV contains paths like:
 'Calc-Training_P_00005_RIGHT_CC_1/1.3.6.../1.3.6.../000001.dcm'
 
 Extract only the case directory name (first part) and search
 for DICOM files within that directory structure.
 
 Args:
 config: ExperimentConfig instance
 csv_files: List of CSV file paths (calc and mass)
 split_name: Name of data split for logging
 
 Returns:
 tuple: (images array, labels array)
 """
 images = []
 labels = []
 skipped_no_path = 0
 skipped_no_file = 0
 skipped_load_error = 0
 processed_cases = set() # Track unique cases to avoid duplicates
 
 for csv_file in csv_files:
 if not csv_file.exists():
 print(f"CSV not found: {csv_file}")
 continue
 
 df = pd.read_csv(csv_file)
 
 # Debug: print available columns
 print(f" Columns in {csv_file.name}: {list(df.columns)[:5]}...")
 
 # Determine which column contains the cropped/ROI image path
 path_col = None
 for col_name in ['cropped image file path', 'ROI mask file path', 'image file path']:
 if col_name in df.columns:
 path_col = col_name
 print(f" Using column: '{path_col}'")
 break
 
 if path_col is None:
 print(f" No valid path column found in {csv_file.name}")
 continue
 
 for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {csv_file.name}"):
 # Get pathology label
 pathology = str(row.get('pathology', '')).upper()
 if 'MALIGNANT' in pathology:
 label = 1
 elif 'BENIGN' in pathology:
 label = 0
 else:
 continue
 
 # Get relative path from CSV
 rel_path = row.get(path_col, None)
 if pd.isna(rel_path) or rel_path is None:
 skipped_no_path += 1
 continue
 
 # Extract case directory name (first part of the path)
 # CSV path: 'Calc-Training_P_00005_RIGHT_CC_1/1.3.6.../1.3.6.../000001.dcm'
 # Required:: 'Calc-Training_P_00005_RIGHT_CC_1'
 case_dir_name = Path(rel_path).parts[0]
 
 # Skip if already processed this case (avoid duplicates)
 case_key = (case_dir_name, label)
 if case_key in processed_cases:
 continue
 processed_cases.add(case_key)
 
 # Find DICOM file in the case directory
 dicom_file = find_dicom_file(case_dir_name, config.dicom_path)
 if dicom_file is None:
 skipped_no_file += 1
 continue
 
 # Load and preprocess image
 image = load_dicom_image(
 dicom_file, 
 target_size=config.image_size,
 use_clahe=config.use_clahe
 )
 
 if image is not None:
 images.append(image)
 labels.append(label)
 else:
 skipped_load_error += 1
 
 print(f" Loaded: {len(images)} images")
 print(f" Skipped (no path): {skipped_no_path}")
 print(f" Skipped (file not found): {skipped_no_file}")
 print(f" Skipped (load error): {skipped_load_error}")
 
 if len(images) == 0:
 return np.array([]), np.array([])
 
 return np.array(images, dtype=np.float32), np.array(labels, dtype=np.float32)


def create_roi_cache(config):
 """
 Create preprocessed ROI cache files.
 
 Args:
 config: ExperimentConfig instance
 """
 print("Creating ROI cache from DICOM files...")
 print(f"DICOM base path: {config.dicom_path}")
 print(f"Path exists: {config.dicom_path.exists()}")
 
 # List contents of DICOM path for debugging
 if config.dicom_path.exists():
 contents = list(config.dicom_path.iterdir())[:5]
 print(f"DICOM directory sample: {[c.name for c in contents]}")
 
 # Process training data
 print("\nProcessing training data...")
 train_csvs = [config.calc_train_csv, config.mass_train_csv]
 train_images, train_labels = preprocess_roi_dataset(config, train_csvs, 'train')
 
 # Process test data
 print("\nProcessing test data...")
 test_csvs = [config.calc_test_csv, config.mass_test_csv]
 test_images, test_labels = preprocess_roi_dataset(config, test_csvs, 'test')
 
 # Split training into train/val
 if len(train_images) > 0:
 val_ratio = config.val_split / (config.train_split + config.val_split)
 
 train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
 train_images, train_labels,
 test_size=val_ratio,
 random_state=RANDOM_SEED,
 stratify=train_labels
 )
 
 # Save cache files
 np.savez(config.cache_path / 'train_cache.npz', images=train_imgs, labels=train_lbls)
 np.savez(config.cache_path / 'val_cache.npz', images=val_imgs, labels=val_lbls)
 np.savez(config.cache_path / 'test_cache.npz', images=test_images, labels=test_labels)
 
 print(f"\nROI cache created successfully:")
 print(f" Training: {len(train_imgs)} samples")
 print(f" Validation: {len(val_imgs)} samples")
 print(f" Test: {len(test_images)} samples")
 else:
 print("\nERROR: No images found. Debugging info:")
 print(f" DICOM path: {config.dicom_path}")
 
 # Try to diagnose
 if config.calc_train_csv.exists():
 df = pd.read_csv(config.calc_train_csv)
 if 'cropped image file path' in df.columns:
 sample_path = df['cropped image file path'].iloc[0]
 case_dir = Path(sample_path).parts[0]
 full_case_path = config.dicom_path / case_dir
 print(f" Sample CSV path: {sample_path}")
 print(f" Extracted case dir: {case_dir}")
 print(f" Case dir exists: {full_case_path.exists()}")
 if full_case_path.exists():
 print(f" Case dir contents: {list(full_case_path.iterdir())[:3]}")


print("ROI preprocessing functions defined")

## 4. Data Loading

Load preprocessed ROI data from cache. If cache does not exist, preprocessing pipeline executes automatically.

In [ ]:
def verify_paths(config):
 """
 Verify required data paths exist.
 
 Args:
 config: ExperimentConfig instance
 
 Returns:
 bool: True if all paths valid
 """
 paths_status = {
 'DICOM directory': config.dicom_path,
 'CSV directory': config.csv_path,
 'Cache directory': config.cache_path
 }
 
 all_valid = True
 print("Path Verification:")
 for name, path in paths_status.items():
 exists = path.exists()
 status = "OK" if exists else "NOT FOUND"
 print(f" {name}: {status}")
 if not exists:
 all_valid = False
 
 return all_valid


def load_cached_data(config):
 """
 Load preprocessed ROI data from cache files.
 
 Args:
 config: ExperimentConfig instance
 
 Returns:
 tuple: (train_images, train_labels, val_images, val_labels, 
 test_images, test_labels)
 """
 cache_files = {
 'train': config.cache_path / 'train_cache.npz',
 'val': config.cache_path / 'val_cache.npz',
 'test': config.cache_path / 'test_cache.npz'
 }
 
 # Check if cache exists
 if not all(f.exists() for f in cache_files.values()):
 print("ROI cache not found. Creating from DICOM files...")
 create_roi_cache(config)
 
 # Verify cache exists after creation attempt
 if not all(f.exists() for f in cache_files.values()):
 raise FileNotFoundError(
 f"ROI cache files not found in {config.cache_path}. "
 "Verify DICOM directory structure."
 )
 
 print("Loading ROI cached data...")
 
 train_data = np.load(cache_files['train'])
 val_data = np.load(cache_files['val'])
 test_data = np.load(cache_files['test'])
 
 train_images, train_labels = train_data['images'], train_data['labels']
 val_images, val_labels = val_data['images'], val_data['labels']
 test_images, test_labels = test_data['images'], test_data['labels']
 
 print("-" * 60)
 print("ROI DATA LOADED FROM CACHE")
 print("-" * 60)
 print(f"Training: {train_images.shape[0]} samples, {train_images.nbytes / 1e6:.1f} MB")
 print(f"Validation: {val_images.shape[0]} samples, {val_images.nbytes / 1e6:.1f} MB")
 print(f"Test: {test_images.shape[0]} samples, {test_images.nbytes / 1e6:.1f} MB")
 print("-" * 60)
 
 return (train_images, train_labels, val_images, val_labels, 
 test_images, test_labels)


# Verify paths and load data
verify_paths(config)

train_images, train_labels, val_images, val_labels, test_images, test_labels = load_cached_data(config)

# Display class distribution
print("Class Distribution:")
for name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
 benign = int((labels == 0).sum())
 malignant = int((labels == 1).sum())
 total = len(labels)
 print(f" {name}: Benign={benign} ({benign/total*100:.1f}%), Malignant={malignant} ({malignant/total*100:.1f}%)")

## 5. Dataset Pipeline

Construct TensorFlow data pipelines with GPU-accelerated augmentation.

In [ ]:
@tf.function
def augment_images(images):
 """
 Apply GPU-accelerated augmentation to image batch.
 
 Args:
 images: Tensor of shape (batch, height, width, channels)
 
 Returns:
 Augmented image tensor
 """
 images = tf.image.random_flip_left_right(images)
 images = tf.image.random_flip_up_down(images)
 images = tf.image.random_brightness(images, max_delta=0.1)
 images = tf.image.random_contrast(images, lower=0.9, upper=1.1)
 return images


def create_generator(images, labels, shuffle=False, seed=None):
 """
 Generator function for tf.data.Dataset.
 
 Args:
 images: numpy array of images
 labels: numpy array of labels
 shuffle: whether to shuffle indices
 seed: random seed for shuffling
 
 Yields:
 (image, label) tuples
 """
 indices = np.arange(len(images))
 if shuffle:
 rng = np.random.default_rng(seed)
 rng.shuffle(indices)
 
 for idx in indices:
 yield images[idx], labels[idx]


def create_dataset(images, labels, config, training=True, 
 shuffle_seed=None, use_reduced_batch=False):
 """
 Create tf.data.Dataset with optional augmentation.
 
 Args:
 images: numpy array of images
 labels: numpy array of labels
 config: ExperimentConfig instance
 training: whether to apply augmentation
 shuffle_seed: seed for shuffling
 use_reduced_batch: use smaller batch size for fine-tuning
 
 Returns:
 tf.data.Dataset instance
 """
 output_signature = (
 tf.TensorSpec(shape=(config.image_size[0], config.image_size[1], 3), 
 dtype=tf.float32),
 tf.TensorSpec(shape=(), dtype=tf.float32)
 )
 
 dataset = tf.data.Dataset.from_generator(
 lambda: create_generator(images, labels, shuffle=training, seed=shuffle_seed),
 output_signature=output_signature
 )
 
 if training:
 dataset = dataset.shuffle(config.shuffle_buffer)
 
 batch_size = config.batch_size_finetune if use_reduced_batch else config.batch_size
 dataset = dataset.batch(batch_size)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_images(x), y),
 num_parallel_calls=config.num_parallel_calls
 )
 
 dataset = dataset.prefetch(config.prefetch_buffer)
 
 return dataset


# Create datasets
train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

print("Dataset pipelines created")
print(f" Training batch size: {config.batch_size}")
print(f" Fine-tuning batch size: {config.batch_size_finetune}")
print(f" Steps per epoch: {len(train_images) // config.batch_size}")

## 6. Class Weight Computation

Compute balanced class weights for handling class imbalance.

In [ ]:
def compute_class_weights(labels, malignant_multiplier=2.5):
 """
 Compute class weights for imbalanced dataset.
 
 Args:
 labels: numpy array of binary labels
 malignant_multiplier: additional weight for malignant class
 
 Returns:
 dict: class weights {0: weight, 1: weight}
 """
 balanced = compute_class_weight(
 class_weight='balanced',
 classes=np.array([0, 1]),
 y=labels
 )
 
 weights = {
 0: balanced[0],
 1: balanced[1] * malignant_multiplier
 }
 
 return weights


# Compute class weights
config.class_weights = compute_class_weights(
 train_labels, 
 config.malignant_weight_multiplier
)

print("Class Weights:")
print(f" Benign (0): {config.class_weights[0]:.3f}")
print(f" Malignant (1): {config.class_weights[1]:.3f}")

## 7. Model Architecture

Build DenseNet-121 based classification model with custom classification head.

In [ ]:
def create_focal_loss(gamma=2.0, alpha=0.7):
 """
 Create focal loss function for handling class imbalance.
 
 Args:
 gamma: focusing parameter
 alpha: class balancing factor
 
 Returns:
 Focal loss function
 """
 @tf.function
 def focal_loss(y_true, y_pred):
 y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
 
 p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - p_t, gamma)
 
 alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
 
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 
 return tf.reduce_mean(alpha_t * focal_weight * ce)
 
 return focal_loss


def build_model(config, seed=42):
 """
 Build DenseNet-121 based classification model.
 
 Args:
 config: ExperimentConfig instance
 seed: random seed for weight initialization
 
 Returns:
 Compiled Keras model
 """
 tf.random.set_seed(seed)
 np.random.seed(seed)
 
 inputs = layers.Input(shape=(config.image_size[0], config.image_size[1], 3))
 
 base_model = DenseNet121(
 include_top=False,
 weights='imagenet',
 input_tensor=inputs,
 pooling=None
 )
 
 base_model.trainable = False
 
 x = base_model.output
 x = layers.GlobalAveragePooling2D()(x)
 
 x = layers.Dense(
 config.dense_units,
 kernel_regularizer=regularizers.l2(config.l2_regularization),
 kernel_initializer='he_normal'
 )(x)
 x = layers.BatchNormalization()(x)
 x = layers.Activation('relu')(x)
 x = layers.Dropout(config.dropout_rate)(x)
 
 outputs = layers.Dense(
 1, 
 activation='sigmoid',
 dtype='float32'
 )(x)
 
 model = models.Model(inputs=inputs, outputs=outputs)
 
 if config.use_focal_loss:
 loss = create_focal_loss(config.focal_gamma, config.focal_alpha)
 else:
 loss = BinaryCrossentropy(label_smoothing=config.label_smoothing)
 
 model.compile(
 optimizer=Adam(learning_rate=config.stage1_lr, clipnorm=config.gradient_clip_norm),
 loss=loss,
 metrics=[
 BinaryAccuracy(name='accuracy'),
 AUC(name='auc'),
 Precision(name='precision'),
 Recall(name='recall')
 ]
 )
 
 return model


def unfreeze_layers(model, num_frozen_layers):
 """
 Unfreeze backbone layers for fine-tuning.
 
 Args:
 model: Keras model
 num_frozen_layers: number of layers to keep frozen
 """
 for layer in model.layers[:num_frozen_layers]:
 layer.trainable = False
 for layer in model.layers[num_frozen_layers:]:
 layer.trainable = True


def get_trainable_summary(model):
 """
 Get summary of trainable parameters.
 
 Args:
 model: Keras model
 
 Returns:
 tuple: (trainable_count, non_trainable_count)
 """
 trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
 non_trainable = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
 return trainable, non_trainable


# Build test model to verify architecture
test_model = build_model(config, seed=42)

trainable, non_trainable = get_trainable_summary(test_model)
print("Model Architecture Summary:")
print(f" Total parameters: {trainable + non_trainable:,}")
print(f" Trainable: {trainable:,}")
print(f" Non-trainable: {non_trainable:,}")
print(f" Dense layer units: {config.dense_units}")

del test_model
gc.collect()

## 8. Training Callbacks

Define callbacks for learning rate scheduling, early stopping, and model checkpointing.

In [ ]:
class WarmupLearningRateScheduler(Callback):
 """
 Learning rate warmup scheduler.
 """
 
 def __init__(self, warmup_epochs, start_lr, target_lr):
 super().__init__()
 self.warmup_epochs = warmup_epochs
 self.start_lr = start_lr
 self.target_lr = target_lr
 
 def on_epoch_begin(self, epoch, logs=None):
 if epoch < self.warmup_epochs:
 lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
 tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)


class MinEpochEarlyStopping(Callback):
 """
 Early stopping with minimum epoch requirement.
 """
 
 def __init__(self, monitor='val_auc', patience=30, min_epochs=50, mode='max'):
 super().__init__()
 self.monitor = monitor
 self.patience = patience
 self.min_epochs = min_epochs
 self.mode = mode
 self.wait = 0
 self.best = -np.inf if mode == 'max' else np.inf
 self.stopped_epoch = 0
 
 def on_epoch_end(self, epoch, logs=None):
 current = logs.get(self.monitor)
 if current is None:
 return
 
 if epoch < self.min_epochs:
 self._update_best(current)
 return
 
 if self._is_improvement(current):
 self._update_best(current)
 self.wait = 0
 else:
 self.wait += 1
 if self.wait >= self.patience:
 self.stopped_epoch = epoch
 self.model.stop_training = True
 print(f"Early stopping triggered at epoch {epoch + 1}")
 
 def _is_improvement(self, current):
 if self.mode == 'max':
 return current > self.best
 return current < self.best
 
 def _update_best(self, current):
 self.best = current


def get_callbacks(config, stage, model_idx):
 """
 Create callbacks for training stage.
 
 Args:
 config: ExperimentConfig instance
 stage: training stage (1, 2, or 3)
 model_idx: ensemble model index
 
 Returns:
 list: Keras callbacks
 """
 callbacks = []
 
 if stage == 1:
 target_lr = config.stage1_lr
 min_epochs = config.min_epochs_stage1
 elif stage == 2:
 target_lr = config.stage2_lr
 min_epochs = config.min_epochs_stage2
 else:
 target_lr = config.stage3_lr
 min_epochs = config.min_epochs_stage3
 
 if stage == 1 and config.use_lr_warmup:
 callbacks.append(WarmupLearningRateScheduler(
 warmup_epochs=config.warmup_epochs,
 start_lr=config.warmup_start_lr,
 target_lr=target_lr
 ))
 
 callbacks.append(ReduceLROnPlateau(
 monitor='val_auc',
 factor=config.lr_reduce_factor,
 patience=config.lr_reduce_patience,
 min_lr=1e-7,
 mode='max',
 verbose=1
 ))
 
 checkpoint_path = config.checkpoint_path / f'model_{model_idx}_stage{stage}_best.h5'
 callbacks.append(ModelCheckpoint(
 str(checkpoint_path),
 monitor='val_auc',
 save_best_only=True,
 mode='max',
 verbose=0
 ))
 
 callbacks.append(MinEpochEarlyStopping(
 monitor='val_auc',
 patience=config.early_stop_patience,
 min_epochs=min_epochs,
 mode='max'
 ))
 
 return callbacks


print("Callback classes defined")

## 9. Training Functions

Define training function with 3-stage progressive unfreezing.

In [ ]:
def train_model(config, model_idx, train_ds, val_ds, train_steps, val_steps):
 """
 Train a single model with 3-stage progressive unfreezing.
 
 Args:
 config: ExperimentConfig instance
 model_idx: ensemble model index
 train_ds: training dataset
 val_ds: validation dataset
 train_steps: steps per training epoch
 val_steps: steps per validation epoch
 
 Returns:
 tuple: (trained model, combined training history)
 """
 print(f"Training Model {model_idx + 1}")
 print("=" * 50)
 
 seed = config.ensemble_seeds[model_idx]
 model = build_model(config, seed=seed)
 
 combined_history = {
 'loss': [], 'val_loss': [],
 'accuracy': [], 'val_accuracy': [],
 'auc': [], 'val_auc': [],
 'precision': [], 'val_precision': [],
 'recall': [], 'val_recall': [],
 'lr': []
 }
 
 # Stage 1: Train head only
 print("Stage 1: Training classification head")
 callbacks = get_callbacks(config, stage=1, model_idx=model_idx)
 
 history1 = model.fit(
 train_ds,
 epochs=config.stage1_epochs,
 steps_per_epoch=train_steps,
 validation_data=val_ds,
 validation_steps=val_steps,
 class_weight=config.class_weights,
 callbacks=callbacks,
 verbose=1
 )
 
 for key in combined_history:
 if key in history1.history:
 combined_history[key].extend(history1.history[key])
 elif key == 'lr':
 combined_history['lr'].extend([float(tf.keras.backend.get_value(model.optimizer.learning_rate))] * len(history1.history['loss']))
 
 # Stage 2: Unfreeze top layers
 print("Stage 2: Fine-tuning top layers")
 unfreeze_layers(model, config.stage2_frozen_layers)
 
 if config.use_focal_loss:
 loss = create_focal_loss(config.focal_gamma, config.focal_alpha)
 else:
 loss = BinaryCrossentropy(label_smoothing=config.label_smoothing)
 
 model.compile(
 optimizer=Adam(learning_rate=config.stage2_lr, clipnorm=config.gradient_clip_norm),
 loss=loss,
 metrics=[
 BinaryAccuracy(name='accuracy'),
 AUC(name='auc'),
 Precision(name='precision'),
 Recall(name='recall')
 ]
 )
 
 trainable, non_trainable = get_trainable_summary(model)
 print(f" Trainable parameters: {trainable:,}")
 
 callbacks = get_callbacks(config, stage=2, model_idx=model_idx)
 
 history2 = model.fit(
 train_ds,
 epochs=config.stage2_epochs,
 steps_per_epoch=train_steps,
 validation_data=val_ds,
 validation_steps=val_steps,
 class_weight=config.class_weights,
 callbacks=callbacks,
 verbose=1
 )
 
 for key in combined_history:
 if key in history2.history:
 combined_history[key].extend(history2.history[key])
 elif key == 'lr':
 combined_history['lr'].extend([float(tf.keras.backend.get_value(model.optimizer.learning_rate))] * len(history2.history['loss']))
 
 # Stage 3: Fine-tune all layers
 print("Stage 3: Full fine-tuning")
 unfreeze_layers(model, 0)
 
 model.compile(
 optimizer=Adam(learning_rate=config.stage3_lr, clipnorm=config.gradient_clip_norm),
 loss=loss,
 metrics=[
 BinaryAccuracy(name='accuracy'),
 AUC(name='auc'),
 Precision(name='precision'),
 Recall(name='recall')
 ]
 )
 
 trainable, non_trainable = get_trainable_summary(model)
 print(f" Trainable parameters: {trainable:,}")
 
 callbacks = get_callbacks(config, stage=3, model_idx=model_idx)
 
 history3 = model.fit(
 train_ds,
 epochs=config.stage3_epochs,
 steps_per_epoch=train_steps,
 validation_data=val_ds,
 validation_steps=val_steps,
 class_weight=config.class_weights,
 callbacks=callbacks,
 verbose=1
 )
 
 for key in combined_history:
 if key in history3.history:
 combined_history[key].extend(history3.history[key])
 elif key == 'lr':
 combined_history['lr'].extend([float(tf.keras.backend.get_value(model.optimizer.learning_rate))] * len(history3.history['loss']))
 
 # Load best weights
 best_checkpoint = config.checkpoint_path / f'model_{model_idx}_stage3_best.h5'
 if best_checkpoint.exists():
 model.load_weights(str(best_checkpoint))
 print(f"Loaded best weights from stage 3")
 
 return model, combined_history


def train_ensemble(config, train_ds, val_ds, train_steps, val_steps):
 """
 Train ensemble of models.
 
 Args:
 config: ExperimentConfig instance
 train_ds: training dataset
 val_ds: validation dataset
 train_steps: steps per training epoch
 val_steps: steps per validation epoch
 
 Returns:
 tuple: (list of models, list of histories)
 """
 ensemble_models = []
 ensemble_histories = []
 
 for i in range(config.ensemble_size):
 model, history = train_model(
 config, i, train_ds, val_ds, train_steps, val_steps
 )
 ensemble_models.append(model)
 ensemble_histories.append(history)
 
 gc.collect()
 tf.keras.backend.clear_session()
 
 if i < config.ensemble_size - 1:
 model = build_model(config, seed=config.ensemble_seeds[i])
 model.load_weights(str(config.checkpoint_path / f'model_{i}_stage3_best.h5'))
 ensemble_models[i] = model
 
 return ensemble_models, ensemble_histories


print("Training functions defined")

## 10. Execute Training

Run the full training pipeline.

In [ ]:
# Calculate steps per epoch
train_steps = len(train_images) // config.batch_size
val_steps = len(val_images) // config.batch_size

# Create datasets for training
train_ds = create_dataset(train_images, train_labels, config, training=True)
val_ds = create_dataset(val_images, val_labels, config, training=False)

print("Training Configuration:")
print(f" Training samples: {len(train_labels)}")
print(f" Validation samples: {len(val_labels)}")
print(f" Steps per epoch: {train_steps}")
print(f" Validation steps: {val_steps}")
print(f" Ensemble size: {config.ensemble_size}")
print(f" Total epochs: {(config.stage1_epochs + config.stage2_epochs + config.stage3_epochs) * config.ensemble_size}")

# Train ensemble
ensemble_models, ensemble_histories = train_ensemble(
 config, train_ds, val_ds, train_steps, val_steps
)

print("Training completed")

## 11. Test-Time Augmentation

Apply test-time augmentation for robust predictions.

In [ ]:
def apply_tta_augmentation(image, augmentation_idx):
 """
 Apply single TTA augmentation.
 
 Args:
 image: input image tensor
 augmentation_idx: index of augmentation to apply
 
 Returns:
 Augmented image tensor
 """
 if augmentation_idx == 0:
 return image
 elif augmentation_idx == 1:
 return tf.image.flip_left_right(image)
 elif augmentation_idx == 2:
 return tf.image.flip_up_down(image)
 elif augmentation_idx == 3:
 return tf.image.rot90(image, k=1)
 elif augmentation_idx == 4:
 return tf.image.rot90(image, k=2)
 elif augmentation_idx == 5:
 return tf.image.rot90(image, k=3)
 elif augmentation_idx == 6:
 return tf.image.flip_left_right(tf.image.rot90(image, k=1))
 elif augmentation_idx == 7:
 return tf.image.flip_up_down(tf.image.rot90(image, k=1))
 return image


def predict_with_tta(models, images, num_augmentations=8, batch_size=16):
 """
 Make predictions with test-time augmentation.
 
 Args:
 models: list of trained models
 images: input images
 num_augmentations: number of TTA augmentations
 batch_size: prediction batch size
 
 Returns:
 numpy array of averaged predictions
 """
 all_predictions = []
 
 for aug_idx in range(num_augmentations):
 augmented_images = np.array([
 apply_tta_augmentation(img, aug_idx).numpy() 
 for img in tf.constant(images)
 ])
 
 model_predictions = []
 for model in models:
 preds = model.predict(augmented_images, batch_size=batch_size, verbose=0)
 model_predictions.append(preds.flatten())
 
 ensemble_pred = np.mean(model_predictions, axis=0)
 all_predictions.append(ensemble_pred)
 
 final_predictions = np.mean(all_predictions, axis=0)
 return final_predictions


print("TTA functions defined")

## 12. Evaluation

Evaluate ensemble on test set with comprehensive metrics.

In [ ]:
def evaluate_model(models, test_images, test_labels, config):
 """
 Comprehensive model evaluation with TTA.
 
 Args:
 models: list of trained models
 test_images: test set images
 test_labels: test set labels
 config: ExperimentConfig instance
 
 Returns:
 dict: evaluation metrics and predictions
 """
 print("Evaluating on test set with TTA...")
 predictions = predict_with_tta(
 models, test_images, 
 num_augmentations=8, 
 batch_size=config.batch_size
 )
 
 # Standard metrics at 0.5 threshold
 predicted_classes = (predictions > 0.5).astype(int)
 
 auc_score = roc_auc_score(test_labels, predictions)
 accuracy = accuracy_score(test_labels, predicted_classes)
 precision = precision_score(test_labels, predicted_classes, zero_division=0)
 recall = recall_score(test_labels, predicted_classes, zero_division=0)
 f1 = f1_score(test_labels, predicted_classes, zero_division=0)
 
 tn, fp, fn, tp = confusion_matrix(test_labels, predicted_classes).ravel()
 specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
 
 # Find optimal threshold
 fpr, tpr, thresholds = roc_curve(test_labels, predictions)
 optimal_idx = np.argmax(tpr - fpr)
 optimal_threshold = thresholds[optimal_idx]
 
 optimal_predicted = (predictions > optimal_threshold).astype(int)
 optimal_accuracy = accuracy_score(test_labels, optimal_predicted)
 optimal_precision = precision_score(test_labels, optimal_predicted, zero_division=0)
 optimal_recall = recall_score(test_labels, optimal_predicted, zero_division=0)
 optimal_f1 = f1_score(test_labels, optimal_predicted, zero_division=0)
 
 tn_opt, fp_opt, fn_opt, tp_opt = confusion_matrix(test_labels, optimal_predicted).ravel()
 optimal_specificity = tn_opt / (tn_opt + fp_opt) if (tn_opt + fp_opt) > 0 else 0
 
 results = {
 'auc': float(auc_score),
 'accuracy': float(accuracy),
 'precision': float(precision),
 'recall': float(recall),
 'sensitivity': float(recall),
 'specificity': float(specificity),
 'f1_score': float(f1),
 'optimal_threshold': float(optimal_threshold),
 'optimal_accuracy': float(optimal_accuracy),
 'optimal_precision': float(optimal_precision),
 'optimal_recall': float(optimal_recall),
 'optimal_specificity': float(optimal_specificity),
 'optimal_f1': float(optimal_f1),
 'confusion_matrix': {
 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
 },
 'optimal_confusion_matrix': {
 'tn': int(tn_opt), 'fp': int(fp_opt), 'fn': int(fn_opt), 'tp': int(tp_opt)
 },
 'predictions': predictions.tolist(),
 'true_labels': test_labels.tolist(),
 'fpr': fpr.tolist(),
 'tpr': tpr.tolist()
 }
 
 return results


print("Evaluation function defined")

In [ ]:
# Load models from checkpoints if needed
loaded_models = []
for i in range(config.ensemble_size):
 model = build_model(config, seed=config.ensemble_seeds[i])
 checkpoint_path = config.checkpoint_path / f'model_{i}_stage3_best.h5'
 if checkpoint_path.exists():
 model.load_weights(str(checkpoint_path))
 loaded_models.append(model)
 print(f"Loaded model {i + 1} from checkpoint")

if loaded_models:
 ensemble_models = loaded_models

# Evaluate
results = evaluate_model(ensemble_models, test_images, test_labels, config)

# Print results
print("=" * 60)
print("TEST SET EVALUATION RESULTS")
print("=" * 60)
print()
print("Standard Threshold (0.5):")
print(f" AUC: {results['auc']:.4f}")
print(f" Accuracy: {results['accuracy']:.4f}")
print(f" Sensitivity: {results['sensitivity']:.4f}")
print(f" Specificity: {results['specificity']:.4f}")
print(f" Precision: {results['precision']:.4f}")
print(f" F1 Score: {results['f1_score']:.4f}")
print()
print(f"Optimal Threshold ({results['optimal_threshold']:.3f}):")
print(f" Accuracy: {results['optimal_accuracy']:.4f}")
print(f" Sensitivity: {results['optimal_recall']:.4f}")
print(f" Specificity: {results['optimal_specificity']:.4f}")
print(f" Precision: {results['optimal_precision']:.4f}")
print(f" F1 Score: {results['optimal_f1']:.4f}")
print()
print("Confusion Matrix (threshold=0.5):")
cm = results['confusion_matrix']
print(f" TN={cm['tn']}, FP={cm['fp']}")
print(f" FN={cm['fn']}, TP={cm['tp']}")

## 13. Visualization

Generate diagnostic plots for model performance analysis.

In [ ]:
def plot_roc_curve(results, save_path=None):
 """Plot ROC curve with AUC."""
 plt.figure(figsize=(8, 8))
 plt.plot(results['fpr'], results['tpr'], 
 color='blue', lw=2, 
 label=f"ROC curve (AUC = {results['auc']:.4f})")
 plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
 plt.xlim([0.0, 1.0])
 plt.ylim([0.0, 1.05])
 plt.xlabel('False Positive Rate', fontsize=12)
 plt.ylabel('True Positive Rate', fontsize=12)
 plt.title('ROC Curve - ROI-Based Classification', fontsize=14)
 plt.legend(loc='lower right', fontsize=11)
 plt.grid(True, alpha=0.3)
 
 if save_path:
 plt.savefig(save_path, dpi=150, bbox_inches='tight')
 plt.show()


def plot_confusion_matrices(results, save_path=None):
 """Plot confusion matrices at standard and optimal thresholds."""
 fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
 cm_std = np.array([
 [results['confusion_matrix']['tn'], results['confusion_matrix']['fp']],
 [results['confusion_matrix']['fn'], results['confusion_matrix']['tp']]
 ])
 
 sns.heatmap(cm_std, annot=True, fmt='d', cmap='Blues', ax=axes[0],
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
 axes[0].set_xlabel('Predicted', fontsize=11)
 axes[0].set_ylabel('Actual', fontsize=11)
 axes[0].set_title('Confusion Matrix (threshold=0.5)', fontsize=12)
 
 cm_opt = np.array([
 [results['optimal_confusion_matrix']['tn'], results['optimal_confusion_matrix']['fp']],
 [results['optimal_confusion_matrix']['fn'], results['optimal_confusion_matrix']['tp']]
 ])
 
 sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', ax=axes[1],
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
 axes[1].set_xlabel('Predicted', fontsize=11)
 axes[1].set_ylabel('Actual', fontsize=11)
 axes[1].set_title(f"Confusion Matrix (threshold={results['optimal_threshold']:.3f})", fontsize=12)
 
 plt.tight_layout()
 if save_path:
 plt.savefig(save_path, dpi=150, bbox_inches='tight')
 plt.show()


def plot_training_history(histories, save_path=None):
 """Plot training history for ensemble."""
 fig, axes = plt.subplots(2, 2, figsize=(14, 10))
 
 for i, history in enumerate(histories):
 label = f'Model {i+1}'
 
 axes[0, 0].plot(history['loss'], label=f'{label} Train', alpha=0.7)
 axes[0, 0].plot(history['val_loss'], label=f'{label} Val', linestyle='--', alpha=0.7)
 
 axes[0, 1].plot(history['auc'], label=f'{label} Train', alpha=0.7)
 axes[0, 1].plot(history['val_auc'], label=f'{label} Val', linestyle='--', alpha=0.7)
 
 axes[1, 0].plot(history['accuracy'], label=f'{label} Train', alpha=0.7)
 axes[1, 0].plot(history['val_accuracy'], label=f'{label} Val', linestyle='--', alpha=0.7)
 
 if 'lr' in history:
 axes[1, 1].plot(history['lr'], label=label, alpha=0.7)
 
 axes[0, 0].set_title('Loss', fontsize=12)
 axes[0, 0].set_xlabel('Epoch')
 axes[0, 0].set_ylabel('Loss')
 axes[0, 0].legend(fontsize=8)
 axes[0, 0].grid(True, alpha=0.3)
 
 axes[0, 1].set_title('AUC', fontsize=12)
 axes[0, 1].set_xlabel('Epoch')
 axes[0, 1].set_ylabel('AUC')
 axes[0, 1].legend(fontsize=8)
 axes[0, 1].grid(True, alpha=0.3)
 
 axes[1, 0].set_title('Accuracy', fontsize=12)
 axes[1, 0].set_xlabel('Epoch')
 axes[1, 0].set_ylabel('Accuracy')
 axes[1, 0].legend(fontsize=8)
 axes[1, 0].grid(True, alpha=0.3)
 
 axes[1, 1].set_title('Learning Rate', fontsize=12)
 axes[1, 1].set_xlabel('Epoch')
 axes[1, 1].set_ylabel('Learning Rate')
 axes[1, 1].legend(fontsize=8)
 axes[1, 1].set_yscale('log')
 axes[1, 1].grid(True, alpha=0.3)
 
 plt.tight_layout()
 if save_path:
 plt.savefig(save_path, dpi=150, bbox_inches='tight')
 plt.show()


# Generate plots
plot_roc_curve(results, save_path=config.results_path / 'roc_curve.png')
plot_confusion_matrices(results, save_path=config.results_path / 'confusion_matrices.png')
plot_training_history(ensemble_histories, save_path=config.results_path / 'training_history.png')

print("Visualizations saved")

## 14. Save Results

Save final model and experiment results.

In [ ]:
# Save final ensemble model (save each member)
for i, model in enumerate(ensemble_models):
 model_path = config.results_path / f'roi_ensemble_model_{i}.h5'
 model.save(str(model_path))
 print(f"Saved model {i + 1} to {model_path}")

# Save complete results
results_with_config = {
 'experiment': 'V12_ROI',
 'description': 'ROI-based classification with DenseNet-121 ensemble',
 'data_source': 'CBIS-DDSM ROI cropped images',
 'configuration': {
 'image_size': list(config.image_size),
 'batch_size': config.batch_size,
 'ensemble_size': config.ensemble_size,
 'stage1_epochs': config.stage1_epochs,
 'stage2_epochs': config.stage2_epochs,
 'stage3_epochs': config.stage3_epochs,
 'l2_regularization': config.l2_regularization,
 'dropout_rate': config.dropout_rate,
 'focal_loss': config.use_focal_loss,
 'focal_gamma': config.focal_gamma,
 'focal_alpha': config.focal_alpha,
 'class_weights': {str(k): float(v) for k, v in config.class_weights.items()} if config.class_weights else None
 },
 'results': {
 'auc': results['auc'],
 'accuracy': results['accuracy'],
 'sensitivity': results['sensitivity'],
 'specificity': results['specificity'],
 'precision': results['precision'],
 'f1_score': results['f1_score'],
 'optimal_threshold': results['optimal_threshold'],
 'optimal_accuracy': results['optimal_accuracy'],
 'optimal_sensitivity': results['optimal_recall'],
 'optimal_specificity': results['optimal_specificity'],
 'optimal_f1': results['optimal_f1']
 },
 'confusion_matrix': results['confusion_matrix'],
 'optimal_confusion_matrix': results['optimal_confusion_matrix']
}

results_path = config.results_path / 'roi_results.json'
with open(results_path, 'w') as f:
 json.dump(results_with_config, f, indent=2)
print(f"Results saved to {results_path}")

# Save training histories
histories_path = config.results_path / 'roi_training_histories.json'
serializable_histories = []
for h in ensemble_histories:
 serializable_histories.append({k: [float(v) for v in vals] for k, vals in h.items()})
with open(histories_path, 'w') as f:
 json.dump(serializable_histories, f, indent=2)
print(f"Training histories saved to {histories_path}")

## 15. Experiment Summary

Final summary of the ROI-based classification experiment.

In [ ]:
print("=" * 70)
print("V12 ROI-BASED CLASSIFICATION EXPERIMENT SUMMARY")
print("=" * 70)
print()
print("Data Configuration:")
print(f" Source: CBIS-DDSM ROI Cropped Images")
print(f" Image Size: {config.image_size[0]}x{config.image_size[1]}")
print(f" Preprocessing: CLAHE enhancement")
print()
print("Model Configuration:")
print(f" Architecture: DenseNet-121")
print(f" Ensemble Size: {config.ensemble_size}")
print(f" Dense Units: {config.dense_units}")
print(f" Regularization: L2={config.l2_regularization}, Dropout={config.dropout_rate}")
print(f" Loss Function: Focal Loss (gamma={config.focal_gamma}, alpha={config.focal_alpha})")
print()
print("Training Configuration:")
print(f" Stage 1: {config.stage1_epochs} epochs (head only)")
print(f" Stage 2: {config.stage2_epochs} epochs (top layers)")
print(f" Stage 3: {config.stage3_epochs} epochs (full fine-tune)")
print()
print("Final Test Results:")
print(f" AUC: {results['auc']:.4f}")
print(f" Accuracy: {results['accuracy']:.4f}")
print(f" Sensitivity: {results['sensitivity']:.4f}")
print(f" Specificity: {results['specificity']:.4f}")
print(f" F1 Score: {results['f1_score']:.4f}")
print()
print("Optimal Threshold Results:")
print(f" Threshold: {results['optimal_threshold']:.3f}")
print(f" Accuracy: {results['optimal_accuracy']:.4f}")
print(f" Sensitivity: {results['optimal_recall']:.4f}")
print(f" Specificity: {results['optimal_specificity']:.4f}")
print()
print("Comparison Notes:")
print(" V11 Full Mammogram AUC: 0.7616")
print(f" V12 ROI-Based AUC: {results['auc']:.4f}")
print()
print("=" * 70)
print("Experiment complete. Results saved to:", config.results_path)
print("=" * 70)